In [0]:

from pyspark.sql import functions as F
import time

# 1. Grande table : 30 millions de transactions
(spark.range(0, 30_000_000)
    .withColumn("compte_id", (F.rand(1) * 2_000_000).cast("long"))
    .withColumn("montant", F.round(F.rand(2) * 500, 2))
    .write.mode("overwrite").saveAsTable("workspace.default.tx"))

# 2. Petite table : 20 000 comptes suspects
(spark.range(0, 20_000).withColumnRenamed("id", "compte_id")
    .withColumn("motif", F.lit("alerte"))
    .write.mode("overwrite").saveAsTable("workspace.default.suspects"))

tx = spark.table("workspace.default.tx")
s = spark.table("workspace.default.suspects")

# 3. Trois strategies imposees par hint
essais = {
    "sort-merge":   tx.join(s.hint("merge"), "compte_id"),
    "shuffle hash": tx.join(s.hint("shuffle_hash"), "compte_id"),
    "broadcast":    tx.join(F.broadcast(s), "compte_id"),
}
for nom, df in essais.items():
    t0 = time.time()
    df.agg(F.count("*"), F.sum("montant")).collect()
    print(f"{nom:13s} {time.time() - t0:6.1f} s")


In [0]:

from pyspark.sql import functions as F
import time

# 1. Grande table : 30 millions de transactions
(spark.range(0, 30_000_000)
    .withColumn("compte_id", (F.rand(1) * 2_000_000).cast("long"))
    .withColumn("montant", F.round(F.rand(2) * 500, 2))
    .write.mode("overwrite").saveAsTable("workspace.default.tx"))

# 2. Petite table : 20 000 comptes suspects
(spark.range(0, 20_000).withColumnRenamed("id", "compte_id")
    .withColumn("motif", F.lit("alerte"))
    .write.mode("overwrite").saveAsTable("workspace.default.suspects"))

tx = spark.table("workspace.default.tx")
s = spark.table("workspace.default.suspects")

# 3. Exécution sans hint
essais = {
    "sans hint":   tx.join(s, "compte_id")
}
for nom, df in essais.items():
    t0 = time.time()
    df.agg(F.count("*"), F.sum("montant")).collect()
    print(f"{nom:13s} {time.time() - t0:6.1f} s")
